# Historical Air Quality Analysis — AirQualityUCI

**Dataset:** UCI Air Quality (De Vito et al.) — hourly measurements from a multisensor device deployed in an Italian city, March 2004 – April 2005.

**Goal:** explore pollutant trends, quantify pollutant–weather relationships, and build a simple predictive model for benzene.

**Tools:** pandas · matplotlib · seaborn · scikit-learn

## 1 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

## 2 · Load and inspect

The CSV uses **semicolons** as separators, **commas** as decimal points, and **−200** as the missing-value sentinel. The file also has two stray empty trailing columns and ~100 blank rows at the end.

In [ ]:
SRC = "../data/raw/AirQualityUCI.csv"

df = pd.read_csv(SRC, sep=";", decimal=",", na_values=["-200", "-200,0"])
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]
df = df.dropna(how="all").reset_index(drop=True)

print("Shape:", df.shape)
df.head()

## 3 · Clean: replace remaining sentinels, build datetime index, rename columns

In [ ]:
# Replace any -200 still hiding (some columns may sneak through with float dtype)
df = df.replace(-200, np.nan)

# Build a proper Datetime index
df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"].str.replace(".", ":", regex=False),
    format="%d/%m/%Y %H:%M:%S",
)
df = df.drop(columns=["Date", "Time"]).set_index("Datetime").sort_index()

# Friendly column names
RENAME = {
    "CO(GT)": "CO_mgm3", "PT08.S1(CO)": "S1_CO",
    "NMHC(GT)": "NMHC_ugm3", "C6H6(GT)": "C6H6_ugm3",
    "PT08.S2(NMHC)": "S2_NMHC", "NOx(GT)": "NOx_ppb",
    "PT08.S3(NOx)": "S3_NOx", "NO2(GT)": "NO2_ugm3",
    "PT08.S4(NO2)": "S4_NO2", "PT08.S5(O3)": "S5_O3",
    "T": "Temp_C", "RH": "RH_pct", "AH": "AbsHum",
}
df = df.rename(columns=RENAME)

print("Period:", df.index.min(), "→", df.index.max())
print("\nMissing % per column:")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))

NMHC is >90 % missing — drop it. Short gaps (≤6 h) get time-interpolated; longer gaps stay NaN so we don't fabricate data.

In [ ]:
df = df.drop(columns=["NMHC_ugm3"])
df_clean = df.interpolate(method="time", limit=6, limit_direction="both")

POLLUTANTS = ["CO_mgm3", "C6H6_ugm3", "NOx_ppb", "NO2_ugm3"]
WEATHER    = ["Temp_C", "RH_pct", "AbsHum"]

df_clean[POLLUTANTS + WEATHER].describe().round(2)

## 4 · Aggregate daily and monthly

In [ ]:
daily   = df_clean[POLLUTANTS + WEATHER].resample("D").mean()
monthly = df_clean[POLLUTANTS + WEATHER].resample("MS").mean()
monthly.head()

## 5 · Visualise — daily time series with rolling mean

In [ ]:
PRETTY = {
    "CO_mgm3": "CO (mg/m³)", "C6H6_ugm3": "Benzene C₆H₆ (µg/m³)",
    "NOx_ppb": "NOx (ppb)", "NO2_ugm3": "NO₂ (µg/m³)",
}
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
for ax, col, c in zip(axes, POLLUTANTS, sns.color_palette("rocket", 4)):
    ax.plot(daily.index, daily[col], color=c, alpha=0.35, lw=0.8, label="Daily mean")
    ax.plot(daily.index, daily[col].rolling(7, min_periods=3).mean(),
            color=c, lw=2, label="7-day rolling")
    ax.set_ylabel(PRETTY[col]); ax.legend(fontsize=8)
axes[0].set_title("Daily pollutant trends, March 2004 – April 2005",
                  fontsize=13, fontweight="bold")
axes[-1].set_xlabel("Date")
plt.tight_layout(); plt.show()

**Observations.** NOx and NO₂ rise sharply from October onwards (cooler, less convective air, more heating). CO and benzene dip in late summer. The dataset ends before the 2005 summer trough.

## 6 · Diurnal cycle — the rush-hour fingerprint

In [ ]:
hourly_profile = df_clean.groupby(df_clean.index.hour)[POLLUTANTS].mean()
fig, ax = plt.subplots(figsize=(10, 5))
for col, c in zip(POLLUTANTS, sns.color_palette("rocket", 4)):
    s = hourly_profile[col] / hourly_profile[col].max()
    ax.plot(s.index, s.values, marker="o", lw=2, color=c, label=PRETTY[col])
ax.set(xticks=range(0, 24, 2), xlabel="Hour of day", ylabel="Normalised concentration")
ax.set_title("Diurnal cycle — pollutants peak with rush hours",
             fontsize=12, fontweight="bold")
ax.legend(); plt.show()

## 7 · Correlation matrix

In [ ]:
corr = df_clean[POLLUTANTS + WEATHER].corr().round(2)
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Pearson correlation: pollutants & weather",
             fontsize=12, fontweight="bold")
plt.show()

**Three blocks pop out:**
1. **Combustion cluster** — CO, benzene, NOx, NO₂ all correlate r ≈ 0.6–0.9 → a pattern consistent with shared combustion-related sources, including road traffic.
2. **Weather** — T and AH correlate +0.66, T and RH correlate −0.58 (standard physics).
3. **Cross terms** — NOx and NO₂ are weakly anti-correlated with temperature (better dispersion in warmer convective air).

## 8 · Weekday vs weekend

In [ ]:
tmp = df_clean.copy()
tmp["DayType"] = np.where(tmp.index.dayofweek < 5, "Weekday", "Weekend")

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col in zip(axes, POLLUTANTS):
    sns.boxplot(data=tmp, x="DayType", y=col, ax=ax,
                palette=["#e74c3c", "#3498db"], showfliers=False)
    ax.set_title(PRETTY[col], fontsize=10); ax.set_xlabel("")
plt.suptitle("Weekday vs weekend distributions", fontweight="bold", y=1.02)
plt.show()

wd = tmp.loc[tmp["DayType"] == "Weekday", "NOx_ppb"].mean()
we = tmp.loc[tmp["DayType"] == "Weekend", "NOx_ppb"].mean()
print(f"NOx mean — Weekday: {wd:.0f} ppb · Weekend: {we:.0f} ppb · drop: {100*(wd-we)/wd:.0f}%")

## 9 · Predictive model — benzene

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

m = df_clean.dropna(subset=POLLUTANTS + WEATHER).copy()
m["hour"], m["dow"], m["month"] = m.index.hour, m.index.dayofweek, m.index.month
features = ["CO_mgm3", "NOx_ppb", "NO2_ugm3",
            "Temp_C", "RH_pct", "AbsHum", "hour", "dow", "month"]
target = "C6H6_ugm3"

X, y = m[features], m[target]
split = int(len(X) * 0.8)
X_tr, X_te, y_tr, y_te = X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]

results = []
for name, mdl in [("Linear Regression", LinearRegression()),
                  ("Random Forest", RandomForestRegressor(
                      n_estimators=120, random_state=0, n_jobs=-1))]:
    mdl.fit(X_tr, y_tr)
    pred = mdl.predict(X_te)
    results.append({"Model": name,
                    "MAE": round(mean_absolute_error(y_te, pred), 2),
                    "R²":  round(r2_score(y_te, pred), 2)})

pd.DataFrame(results)

In [ ]:
rf = RandomForestRegressor(n_estimators=120, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
pred = rf.predict(X_te)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(m.index[split:], y_te.values, color="#34495e", lw=1, alpha=0.8, label="Actual")
ax.plot(m.index[split:], pred,        color="#e67e22", lw=1, alpha=0.8, label="Predicted")
ax.set_title("Benzene — actual vs predicted (Random Forest)", fontweight="bold")
ax.set_ylabel("C₆H₆ (µg/m³)"); ax.legend(); plt.show()

# Feature importance
fi = pd.Series(rf.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
fi.plot(kind="barh", ax=ax, color=sns.color_palette("rocket", len(fi)))
ax.set_title("Random-Forest feature importance", fontweight="bold")
ax.set_xlabel("Relative importance"); plt.show()

## 10 · Conclusions

| Theme | Finding |
| --- | --- |
| **Sources** | CO, benzene, NOx and NO₂ form a strong combustion-related correlation cluster (r = 0.6–0.9), consistent with shared sources such as road traffic. |
| **Diurnal** | Twin peaks at 08:00 and 19:00; trough at 05:00; evening peak the larger of the two. |
| **Seasonal** | Winter peaks for NOx/NO₂; summer dip for CO/benzene. |
| **Weekly** | NOx is ~27 % lower at weekends, a pattern consistent with reduced traffic activity. |
| **Health** | Annual benzene mean (~10 µg/m³) is several × WHO reference and 2× the EU annual limit (5 µg/m³). |
| **Modelling** | A Random Forest predicts hourly benzene at R² ≈ 0.77 from CO + NOx + NO₂ + weather + hour-of-day, showing that co-located pollutant and weather variables contain substantial predictive signal for benzene in this dataset. |

### Limitations
- Single site, 13 months — not enough for inter-annual trends or policy-impact assessment.
- NMHC dropped (>90 % missing).
- Correlation does not equal causation; weather and traffic seasonality are confounded.